# Lab 2 — Global Sensitivity Analysis: Sobol Indices and Feature Scoring

## Background

**Global sensitivity analysis (GSA)** quantifies how much each uncertain input parameter
contributes to the variance of model outputs, across the entire uncertainty space. This is
different from local sensitivity analysis (which perturbs one parameter at a time near a
nominal value).

### Three methods covered in this lab

| Method | Basis | Key output | Computational cost |
|--------|-------|-----------|-------------------|
| Linear regression (OLS) | Linear model fit | Standardised coefficients, R² | Low |
| **Sobol indices** | Variance decomposition | S1 (first-order), ST (total-order) | High (N·(2k+2) runs) |
| **Feature scoring** (Extra Trees) | Machine learning | Feature importance | Moderate |

### Sobol indices explained

Given $k$ uncertain inputs $X_1, \ldots, X_k$ and an output $Y$:

$$S_i = \frac{\text{Var}[\mathbb{E}(Y | X_i)]}{\text{Var}[Y]}, \qquad
S_{T_i} = 1 - \frac{\text{Var}[\mathbb{E}(Y | \mathbf{X}_{\sim i})]}{\text{Var}[Y]}$$

- **$S_i$ (first-order)**: fraction of variance explained by $X_i$ alone.
- **$S_{T_i}$ (total-order)**: fraction explained by $X_i$ including all interactions.
- $S_{T_i} - S_i$ estimates the contribution of interaction terms.

Sobol requires $N(2k+2)$ model evaluations (for $k$ inputs and baseline $N$ samples).
With $k=5$ and $N=1000$, that is $12\,000$ runs. The **Saltelli** sampler in the workbench
handles this automatically.

### Feature scoring

Extra Trees is an ensemble of extremely randomised decision trees. Each tree recursively
partitions the input space to predict the output. The **feature importance** for input $X_i$
is the average reduction in impurity (variance for regression) across all splits on $X_i$.
It approximates Sobol's total-order effects at a fraction of the computational cost.

### What you will do

1. Set up the lake model and run a Latin Hypercube sample.
2. Fit a linear regression model — establish whether linear effects dominate.
3. Run Sobol analysis under three fixed release policies and compare sensitivities.
4. Analyse convergence of Sobol indices with sample size.
5. Apply Extra Trees feature scoring and visualise per-policy importances as heatmaps.

## 1. Model setup and imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('white')

import statsmodels.api as sm
from SALib.analyze import sobol as sobol_analyze

from lakemodel_function import lake_problem
from ema_workbench import (Model, RealParameter, ScalarOutcome,
                           Sample, perform_experiments, MultiprocessingEvaluator,
                           ema_logging, Samplers)
from ema_workbench.em_framework.salib_samplers import get_SALib_problem
from ema_workbench.analysis import feature_scoring
from ema_workbench.analysis.scenario_discovery_util import RuleInductionType

ema_logging.log_to_stderr(ema_logging.INFO)

# --- Lake model ---
lake_model = Model('lakeproblem', function=lake_problem)
lake_model.time_horizon = 100

lake_model.uncertainties = [
    RealParameter('mean',  0.01,  0.05),
    RealParameter('stdev', 0.001, 0.005),
    RealParameter('b',     0.1,   0.45),
    RealParameter('q',     2.0,   4.5),
    RealParameter('delta', 0.93,  0.99),
]
lake_model.levers = [RealParameter(f'l{i}', 0, 0.1)
                     for i in range(lake_model.time_horizon)]
lake_model.outcomes = [
    ScalarOutcome('max_P'),
    ScalarOutcome('utility'),
    ScalarOutcome('inertia'),
    ScalarOutcome('reliability'),
]

## 2. Linear regression — does a linear model capture output variance?

Linear regression fits a model $\hat{Y} = \beta_0 + \sum_i \beta_i X_i$ to the simulation
output. The **R²** (coefficient of determination) tells us what fraction of output variance
the linear model captures. If R² is low, non-linear effects dominate and we need Sobol or
feature scoring.

We use **Latin Hypercube Sampling (LHS)** for this step, which gives better coverage of the
input space than random Monte Carlo with the same number of samples.

In [ ]:
n_lhs = 1000

# Run LHS experiments with a no-release policy
no_release = Sample('no release', **{l.name: 0 for l in lake_model.levers})
exp_lhs, out_lhs = perform_experiments(
    lake_model, scenarios=n_lhs,
    policies=no_release,
    uncertainty_sampling=Samplers.LHS
)

In [ ]:
# Extract the final value of reliability (the last time step) from the scalar outcome
# ScalarOutcome returns a 1-D array directly; no time index needed
reliability_lhs = out_lhs['reliability']
max_P_lhs       = out_lhs['max_P']
utility_lhs     = out_lhs['utility']

# Build regressor matrix: drop non-uncertainty columns
X = exp_lhs.drop(columns=['model', 'policy'] +
                 [l.name for l in lake_model.levers])
X_const = sm.add_constant(X.astype(float))

# --- Fit OLS for reliability ---
est = sm.OLS(reliability_lhs, X_const).fit()
print(est.summary())

**Interpretation:** Look at R² — if it is much below 0.8, the linear model does not capture
most of the variance, implying important non-linear effects or interactions. The `P>|t|`
column shows which parameters are statistically significant. Parameters with p > 0.05
do not have a detectable linear effect on the output.

## 3. Sobol analysis under three release policies

Sobol indices require a special **Saltelli** sample. The workbench takes care of generating
$N(2k+2)$ samples when you specify `uncertainty_sampling=Samplers.SOBOL` and a baseline N.

We test **three fixed policies**:
- `'0'` — zero release (no anthropogenic pollution)
- `'0.05'` — moderate constant release
- `'0.1'` — maximum constant release

Running all three policies in a single `perform_experiments` call is efficient: the uncertainty
sample is generated once, and all policies are evaluated against the same set of uncertainty draws.

In [ ]:
# Define three fixed release policies
policies = [
    Sample('0',    **{l.name: 0.0  for l in lake_model.levers}),
    Sample('0.05', **{l.name: 0.05 for l in lake_model.levers}),
    Sample('0.1',  **{l.name: 0.1  for l in lake_model.levers}),
]

# Sobol requires N*(2k+2) evaluations; the workbench computes this automatically.
# With n_sobol=128 and k=5 inputs: 128 * 12 = 1536 evaluations per policy.
n_sobol = 128

with MultiprocessingEvaluator(lake_model, n_processes=-2) as evaluator:
    experiments_sobol, results_sobol = evaluator.perform_experiments(
        n_sobol, policies, uncertainty_sampling=Samplers.SOBOL
    )

# Build SALib problem definition from the workbench uncertainties
problem = get_SALib_problem(lake_model.uncertainties)
print('SALib problem:', problem)

### 3.1 Compute and visualise Sobol indices per policy

For each policy, we extract the reliability values and run `sobol.analyze`. We then plot
the first-order (S1) and total-order (ST) effects side by side with confidence intervals.

- **S1 ≈ ST**: parameter acts mainly through direct (linear/nonlinear) effects, little interaction.
- **ST >> S1**: parameter has strong interactions with other inputs.

In [ ]:
sobol_results = {}
for policy_name in experiments_sobol['policy'].unique():
    mask = experiments_sobol['policy'] == policy_name
    y = results_sobol['reliability'][mask]
    indices = sobol_analyze.analyze(problem, y, calc_second_order=True,
                                    print_to_console=False)
    sobol_results[policy_name] = indices

# --- Plot S1 and ST for each policy ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (policy_name, indices) in zip(axes, sobol_results.items()):
    Si_df = pd.DataFrame(
        {'S1': indices['S1'], 'ST': indices['ST']},
        index=problem['names']
    )
    err = pd.DataFrame(
        {'S1': indices['S1_conf'], 'ST': indices['ST_conf']},
        index=problem['names']
    )
    Si_df.plot.bar(yerr=err.values.T, ax=ax, capsize=4)
    ax.set_title(f'Policy = {policy_name}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

axes[0].set_ylabel('Sobol index')
fig.suptitle('First-order (S1) and Total-order (ST) Sobol indices — reliability outcome',
             fontsize=12)
plt.tight_layout()
plt.show()

### 3.2 Convergence of Sobol indices

Sobol indices are estimated from finite samples and may not be stable at low sample sizes.
We check convergence by computing ST at progressively larger sub-samples.

A stable estimate (flat line) indicates sufficient sample size; drifting lines suggest
more samples are needed.

In [ ]:
# Use policy '0' for the convergence analysis
mask_0 = experiments_sobol['policy'] == '0'
Y_full = results_sobol['reliability'][mask_0]

# Vary sub-sample size and recompute ST
# Step sizes must be multiples of 2*(D+2)=12 (Sobol block size with calc_second_order=True)
n_total = len(Y_full)
block = 2 * problem["num_vars"] + 2   # 12 for 5 vars (Saltelli formula)
step_size = max(block, (n_total // 20 // block) * block)
sample_sizes = [n for n in range(step_size, n_total + 1, step_size) if n % block == 0]

convergence_data = pd.DataFrame(index=problem['names'], columns=sample_sizes)
for n in sample_sizes:
    idx = sobol_analyze.analyze(problem, Y_full[:n], calc_second_order=True,
                                print_to_console=False)
    convergence_data[n] = idx['ST']

fig, ax = plt.subplots(figsize=(8, 4))
convergence_data.T.plot(ax=ax)
ax.set_xlabel('Sample size')
ax.set_ylabel('Total-order index ST')
ax.set_title('Convergence of Sobol ST — policy = 0, reliability outcome')
ax.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## 4. Feature scoring with Extra Trees

**Extra Trees** (Extremely Randomised Trees) is a fast, non-parametric alternative to Sobol.
It fits a random forest and uses the average impurity decrease at each split as a feature
importance score.

Advantages over Sobol:
- Lower computational cost (uses the LHS or Sobol sample directly).
- Handles non-linearity and interactions automatically.
- Can analyse all outcomes at once.

We apply `get_feature_scores_all` to produce a feature importance matrix for each policy
and visualise the results as heatmaps.

In [ ]:
# Drop lever columns — we only want to score the uncertain parameters
cleaned_experiments = experiments_sobol.drop(
    columns=[l.name for l in lake_model.levers]
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, policy_name in zip(axes, experiments_sobol['policy'].unique()):
    mask = experiments_sobol['policy'] == policy_name
    subset_exp = cleaned_experiments[mask]
    subset_res = {k: v[mask] for k, v in results_sobol.items()}

    scores = feature_scoring.get_feature_scores_all(subset_exp, subset_res)

    sns.heatmap(scores, annot=True, fmt='.2f', cmap='viridis',
                vmin=0, vmax=1, ax=ax, cbar=ax is axes[-1])
    ax.set_title(f'Policy = {policy_name}')
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Extra Trees feature importance — all outcomes', fontsize=12)
plt.tight_layout()
plt.show()

### 4.1 Convergence of feature scores

Just as with Sobol, feature scores become more stable with more samples. Here we repeat
the scoring at increasing sample sizes to assess stability.

In [ ]:
# Use policy '0' sample for convergence
mask_0 = cleaned_experiments['policy'] == '0'
exp_sub  = cleaned_experiments[mask_0]
res_sub  = {k: v[mask_0] for k, v in results_sobol.items()}
n_total  = len(exp_sub)

combined_scores = []
step = max(50, n_total // 15)
for j in range(step, n_total, step):
    sc = feature_scoring.get_feature_scores_all(
        exp_sub.iloc[:j], {k: v[:j] for k, v in res_sub.items()}
    )
    sc_reliability = sc['reliability'].rename(j)
    combined_scores.append(sc_reliability)

combined_scores = pd.concat(combined_scores, axis=1, sort=True)

fig, ax = plt.subplots(figsize=(8, 4))
combined_scores.T.plot(ax=ax)
ax.set_xlabel('Sample size')
ax.set_ylabel('Feature importance score')
ax.set_title('Convergence of Extra Trees feature scores — reliability, policy = 0')
ax.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

## Summary and comparison

| Method | Lake model dominant parameters (reliability) | Notes |
|--------|----------------------------------------------|-------|
| Linear regression | Low R² — linear model inadequate | Confirms non-linear effects |
| Sobol ST | `b`, `q` dominant; small role for `mean` | Gold standard; computationally expensive |
| Feature scoring | `b`, `q` dominant; consistent with Sobol | Fast, converges with fewer samples |

**Key insight:** The lake's natural removal rate (`b`) and recycling exponent (`q`) dominate
reliability across all release policies. The discount rate (`delta`) dominates the utility
outcome. Inertia is policy-determined, not uncertainty-driven.

In **Lab 3** you will use these insights to set up scenario discovery with PRIM, focusing on
the conditions under which reliability is lowest.